In [1]:
import pandas as pd
import numpy as np
import itertools

### Add players and update team size here:

In [2]:
# Change each week
PLAYERS = (
    "Charlie",
    "Alasdair",
    "Joseph",
    "Dom",
    "Mo",
    "Harry",
    "Louis",
    "James R",
    "Ellis",
    "Kieran",
    "Jon",
    "Oliver",
    "Dan M",
    "Danny"
)

# Change
TEAMSIZE = 7

teams = list(itertools.combinations(PLAYERS, TEAMSIZE))
print(len(teams))

3432


Finds win percentage from csv with football results in, updated weekly

This then assigns ranks to players with best win rates (higher rank is better).

In [3]:
games = pd.read_csv("football.csv", index_col="Player")

win_dict = {}
loss_dict = {}
win_rates_dict = {}
for player in PLAYERS:
    if player not in games.index.values:
        win_dict.update({player: 0.0})
        loss_dict.update({player: 0.0})
        win_rates_dict.update({player: 0.0})
        continue
    results = games.loc[player]
    if len(results.values) == 1:
        if results.values == "W":
            win_dict.update({player: 1.0})
            loss_dict.update({player: 0.0})
            continue
        else:
            win_dict.update({player: 0.0})
            loss_dict.update({player: 1.0})
            continue
    else:
        wins = results[results["Result"] == "W"]
        losses = results.loc[results["Result"] == "L"]
        win_dict.update({player: wins.count().sum()})
        loss_dict.update({player: losses.count().sum()})
        win_rates_dict.update({player: wins.count().sum()/results.count().sum()})

WEIGHT_CONST = np.mean(list({n: win_dict[n]+loss_dict[n] for n in win_dict}.values()))
weighted_dict = {k: (win_dict[k]+WEIGHT_CONST*np.mean(list(win_rates_dict.values())))/(loss_dict[k]+win_dict[k]+WEIGHT_CONST) for k in win_dict}
df_win_rates = pd.Series(weighted_dict).to_frame()
df_win_rates.index = df_win_rates.index.rename("Player")
df_win_rates = df_win_rates.rename({0: "Rank"}, axis=1)
df_win_rates

,Rank
Player,
Charlie,0.519365
Alasdair,0.528497
Joseph,0.421552
Dom,0.481502
Mo,0.430720
Harry,0.541890
Louis,0.520745
James R,0.498315
Ellis,0.550926


Finds the 'match rating' for each set of 2 teams, balanced teams are at 0 with more unbalanced being a high rating value

In [4]:
# track index
i = 0
team_dict = {}
for team1 in teams:
    team1_rank = 0
    team2_rank = 0
    team2 = list(PLAYERS)

    for player in team1:
        team1_rank += df_win_rates.loc[player].sum()
        team2.remove(player)

    for player in team2:
        team2_rank += df_win_rates.loc[player].sum()

    match_rating = team1_rank - team2_rank
    team_dict.update({i: match_rating})
    i += 1

team_df = pd.Series(team_dict).to_frame()
team_df = team_df.rename({0: "Match Rating"}, axis=1)
team_df = team_df.abs()
final_games = team_df[team_df["Match Rating"] == team_df["Match Rating"].min()]
final_games

,Match Rating
1621,0.000244
1810,0.000244


For multiple matches of the same rating, finds a random team A from all teams list.

In [5]:
import random

random_team_index = random.choice(final_games.index.values)
random_team_index

1810

Displays final teams

In [6]:
final_team_a = list(teams[random_team_index])
final_team_b = list(PLAYERS)
for player in final_team_a:
    final_team_b.remove(player)
print(f"Team A: {final_team_a}, Team B:{final_team_b}")

Team A: ['Alasdair', 'Joseph', 'Dom', 'Harry', 'Louis', 'Ellis', 'Danny'], Team B:['Charlie', 'Mo', 'James R', 'Kieran', 'Jon', 'Oliver', 'Dan M']


In [7]:
print(len(final_games.index.values))
for team_idx in final_games.index.values:
    team_a = list(teams[team_idx])
    team_b = list(PLAYERS)
    for player in team_a:
        team_b.remove(player)
    print(f"Team A: {team_a}, Team B:{team_b}")

2
Team A: ['Charlie', 'Mo', 'James R', 'Kieran', 'Jon', 'Oliver', 'Dan M'], Team B:['Alasdair', 'Joseph', 'Dom', 'Harry', 'Louis', 'Ellis', 'Danny']
Team A: ['Alasdair', 'Joseph', 'Dom', 'Harry', 'Louis', 'Ellis', 'Danny'], Team B:['Charlie', 'Mo', 'James R', 'Kieran', 'Jon', 'Oliver', 'Dan M']
